# Exercise 4
## Map making
*** 

### Question
How can I plot a map, showing the locations of some samples/data?

### Objectives
1. Explore the syntax for plotting a simple map using Cartopy.
2. Use more advanced syntax for plotting a local area with polygon overlays.
3. Learn how to loop over files to read multiple files into Python.
4. See regular expressions in action to separate files with separate chunks of data.
5. Learn to plot spatial data onto a map.

**Revise:**
- Reading in data;
- Data plotting;
- Defining functions.

### Independent coding
Make a map of the continental US's earthquakes from January to September 2019.

> **Health warning:** Plotting maps in Python is just hard. It's a bit awkward, and the syntax is not particularly intuitive. Don't worry if you struggle with this. The exercise does however use lots of elements of coding you have already covered, so it is useful to see these in a new context, even if the map plotting commands are hard to remember.

This exercise uses the Cartopy library, which doesn't come with Anaconda. If `import cartopy` gives you a `ModuleNotFoundError`, follow the _Install Cartopy_ step in the setup instructions, then click **Restart** and try again.

**Run this cell first.** It loads `check()`, which we use to check your answers to the **Your turn** tasks: fill in the `___` blanks, run the cell, and `check()` tells you whether you're right.

In [ ]:
from check import check, ___

## 4.1 Make a simple map
***

Plotting spatial data is a common task in the Geosciences.  There are several tools available in Python to aid with this, including a spatial version of Pandas __[GeoPandas](https://geopandas.org)__, and a high-level interface for making maps __[Geoplot](https://residentmario.github.io/geoplot/index.html)__.  However, here we are going to use __[Cartopy](https://cartopy.readthedocs.io/stable/)__, which is the machinery underlying geoplot.

Let's begin by plotting the globe.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtckr
#this is the import for cartopy, and we are importing its coordinate reference system tool as ccrs
# the coordinate reference system tells us what type of map projection we are making
# i.e., we need some mathematical transformation to tell us how to convert the surface of a sphere into a flat plot.
import cartopy.crs as ccrs

#let's set up a figure to plot into, in a similar way to what we have done previously
# we are just specifying the size up front here
fig = plt.figure(figsize=(12, 6))

#now, we add a subplot to our figure, i.e., place a panel within the figure
# the '1,1,1' just indicates we have one subplot
# with ccrs.XXXXX we are adding the map projection, here we use the well known Mercator projection
ax = fig.add_subplot(1, 1, 1,
                         projection=ccrs.Mercator())

#let's add coastlines and specify the resolution we want
# note: we have little choice of resolution, 110m (1:110e6 scale) is the coarsest we can get, we would have to load our
#       coastline file to change this.
ax.coastlines(resolution='110m')

#we can add a 'stock' background image just to make things look nicer.
ax.stock_img();

#and we can place some gridlines on
gl = ax.gridlines(draw_labels=True, linestyle=':', color='k');
#...and let's limit which sides the labels plot on
gl.top_labels = False
gl.right_labels = False
#finally, placing the latitude markers every 30 degrees
gl.ylocator = mtckr.FixedLocator([-90, -60, -30, 0, 30, 60, 90])

Notice how the Mercator projection distorts the whole globe: Greenland and the British Isles look far bigger than they are, compared with the equatorial landmasses.  This is because the Mercator projection is __[not an equal-area projection](https://en.wikipedia.org/wiki/Map_projection#Equal-area)__.

Let's try a projection that does preserve area, albeit at the expense of other distortions: the __[Mollweide](https://en.wikipedia.org/wiki/Mollweide_projection)__ projection.

In [ ]:
#same as before
fig = plt.figure(figsize=(12, 6))

#now we choose the Mollweide projection
ax = fig.add_subplot(1, 1, 1,
                         projection=ccrs.Mollweide())

#let's add coastlines and specify the resolution we want
# note: we have little choice of resolution, 110m is the coarsest we can get, we would have to load our
#       coastline file to change this.
ax.coastlines(resolution='110m')

#we can add a 'stock' background image just to make things look nicer.
ax.stock_img();

#and we can place some gridlines on
gl = ax.gridlines(draw_labels=True, linestyle=':', color='k');

> **For the curious:** the GMT manual puts it bluntly: on a Mercator map "the Former Soviet Union looks much bigger than Africa or South America. One may wonder whether this illusion has had any influence on U.S. foreign policy."  __[This article](https://www.bostonglobe.com/metro/2017/03/16/north-america-really-bigger-than-africa-this-map-sets-things-straight/lK52K7aKYFpQ3b8ujJj6LP/story.html)__ discusses what map projections mean in education.

**Your turn:** copy the Mollweide cell above into the empty cell below, and change it to use the Robinson projection, `ccrs.Robinson()`.  Where are the distortions now?

In [ ]:
# your code here

This looks much better; the UK is now restored as a small island on the edge of Europe.  

Another potential annoyance of the maps above is that whilst they get all continental area into the picture, they chop the Pacific in half.  This can easily be changed by specifying the central longitude.

> **Predict:** what will the cell below give? Decide, then run it to see if you were right.

In [ ]:
#same as before
fig = plt.figure(figsize=(12, 6))

#notice now that we are passing the Mollweide projection a 'central_longitude' argument
ax = fig.add_subplot(1, 1, 1,
                         projection=ccrs.Mollweide(central_longitude=-150))

#..same as before
ax.coastlines(resolution='110m')
ax.stock_img();
gl = ax.gridlines(draw_labels=False, linestyle=':', color='k');

**Fix the bug:** this is meant to put a red star on Cambridge (52.2°N, 0.12°E), but the star ends up in the Indian Ocean.  Find the mistake and fix it.

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Mercator())
ax.set_extent([-30, 60, -10, 70], crs=ccrs.PlateCarree())
ax.coastlines(resolution='110m')

#Cambridge is at 52.2 N, 0.12 E
star, = ax.plot(52.2, 0.12, 'r*', markersize=15, transform=ccrs.PlateCarree())

check('cambridge', star.get_xydata()[0])

**Your turn:** make a map centred on New Zealand (about 175°E), using any projection you like.

In [ ]:
# your code here

Note that there are other powerful tools for making maps. One of the best known is __[GMT](https://github.com/GenericMappingTools)__, which also has a Python interface, __[PyGMT](https://www.pygmt.org/)__. These are beyond the scope of these practicals, so let's persevere with Cartopy for now!

## 4.2 Make a more advanced map
***

### 4.2.1 Getting started with the map

Let's now aim to plot some data onto a map.  We are going to revisit the dataset of Icelandic eruptions we investigated in __[Exercise 3](Exercise3.ipynb)__.  

First however, we should focus on producing a map of Iceland and plotting on one of its key features: glaciers.  

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtckr
import cartopy as ctp
import cartopy.crs as ccrs

fig = plt.figure(figsize=(12, 6))

#we will use Mercator for this, as for a small region of the globe the area distortion will not be significant
# we need to shift the central longitude onto Iceland
ax = fig.add_subplot(1, 1, 1,
                         projection=ccrs.Mercator(central_longitude=-15))
#we also want to limit the plot area to Iceland
ax.set_extent([-25, -13, 63, 67.], crs=ccrs.Geodetic())

#let's add coastlines and specify the resolution we want
# note: we now want higher resolution, and the highest available to us is 1:10 million (Natural Earth's '10m' scale).  
# we are also going to have to set up the 'feature' to be plotted and then specifically plot that 
#first setup the coastlines
feature = ctp.feature.NaturalEarthFeature(
    name='coastline', category='physical',
    scale='10m',
    edgecolor='#000000', facecolor='lightgrey')
#...then plot, using zorder=0 so that everything else we add (except the gridlines) is drawn on top of the land
ax.add_feature(feature, zorder=0)

#and we can place some gridlines on, this time I am placing them behind the landmass
# using 'zorder' to indicate where I want the element to appear in the vertical 'stack' of things going onto the plot
gl = ax.gridlines(draw_labels=True, linestyle=':', color='k', zorder=-1);
#...and let's limit which sides the labels plot on
gl.top_labels = False
gl.right_labels = False

Notice one important feature of how we communicated the 'coordinate reference system' to Cartopy: whenever we provided values that were 'raw' latitudes and longitudes (which we did to constrain the plot extent) we specified that the coordinate reference system was `ccrs.Geodetic()`.  The other option that achieves nearly identical results is to use `ccrs.PlateCarree()`.  You will see that we will need to use both projection types to plot lat-lon data.

### 4.2.2 Plotting on ice caps: polygon overlays and reading in multiple files

Now, Iceland wouldn't be Iceland without its ice caps.  To plot on the ice caps we need to loop over the several input files inside `data/jokull/` that contain the xy coordinates of the ice caps, and then transform these into an object that can be plotted (_jökull_ means glacier in Icelandic).  This latter task we perform using the `shapely` library, which turns each list of coordinates into a polygon (more on this powerful library __[here](https://shapely.readthedocs.io/en/stable/manual.html)__).  Remember to look inside those input files to convince yourself something sensible is going to happen when we try and plot them.

Let's write a function to do the reading in and conversion to polygons, so that we can call it each time we remake the plot.

In [ ]:
#we are going to need to find files on the computer, so we use 'Path' from the pathlib library.
# Path builds file locations that work on Windows, macOS and Linux, and can search folders for us.
from pathlib import Path
#Pandas makes everything better
import pandas as pd

#we import the Polygon tool from Shapely's geometry library
# we are going to convert the long-lat points of the ice caps into 'polygon' objects that
# can be plotted nicely on our map
from shapely.geometry import Polygon

#define our function
def poly_jokull():
    """Read the ice cap outlines in data/jokull, and return them as a list of shapely Polygons."""

    #we want a list of all the files in 'data/jokull' that end with '.xy'.
    # Path('data/jokull') is the folder, and .glob('*.xy') finds every file in it matching
    # the pattern '*.xy', where '*' means 'any characters'. sorted() puts them in alphabetical order.
    files = sorted(Path('data/jokull').glob('*.xy'))

    #now we have a list of all the files we would like to read them in, create a polygon object
    # and loop over all the input file names
    #we will store our polygons in a list called 'polys'
    polys = []
    for f in files:
        #read in using pandas, you might have noticed that the jokull files have a first line
        # starting with '>', this 'header' needs to be skipped, so we tell pandas that it is a 'comment'
        # so that it gets ignored.
        dxy = pd.read_csv(f, comment='>', sep=' ', header=None)

        #now we use the shapely function to create a new polygon and add this to our list of polygons
        # the lon and lat data needs to be passed separately, but 'zipped' together
        polys.append(Polygon(zip(dxy[0], dxy[1])))

    #finally, we send our list of polygons back
    return polys

Now we have written our function, let's call it, obtain our list of polygon objects, and then plot them.

In [ ]:
#call our new function to obtain all the ice cap locations.
poly_geoms = poly_jokull()

#----------- same as before -----------------------
fig = plt.figure(figsize=(12, 6))
ax = fig.add_subplot(1, 1, 1,
                         projection=ccrs.Mercator(central_longitude=-15))
ax.set_extent([-25, -13, 63, 67.], crs=ccrs.Geodetic())

feature = ctp.feature.NaturalEarthFeature(
    name='coastline', category='physical',
    scale='10m',
    edgecolor='#000000', facecolor='lightgrey')
ax.add_feature(feature, zorder=0)

gl = ax.gridlines(draw_labels=True, linestyle=':', color='k', zorder=-1);
gl.top_labels = False
gl.right_labels = False
#----------- end of same as before -----------------------

#now, we need to loop over all of those polygons plotting them, which we do with the
# 'add_geometries' method of our axes object
# note, that here we need to use PlateCarree() as the coordinate reference system.
# and we set 'facecolor=white', because ice caps are white.
for p in poly_geoms:
    x = ax.add_geometries([p], crs=ccrs.PlateCarree(), facecolor='white')

**Your turn:** how many ice cap outlines did `poly_jokull()` give back?

In [ ]:
n_icecaps = ___
check('ice caps', n_icecaps)

### 4.2.3 Reading in awkward files

Finally, let's add Iceland's volcanic rift zones.  This is a trickier problem.  We have been lucky so far with the files we have had to read in: they have all had a nice format that Pandas can accept.  Our next task is the first where the file needs a bit more pre-processing before it is useful to us.  Unfortunately, needing to pre-process input files is _very_ common!  

We could do this manually, but it is useful to see how to use some of the lower-level file reading machinery in Python, because when you have a large file you don't want to be having to go through all of it yourself.  Potentially then having to repeat the exercise if the input file gets updated...

Like with the ice caps, we are going to define a function to do the reading in and conversion to polygons for us.  Start by opening up the `fisswarms_fil.xy` file, so you can see the problems we face with this file. (Perhaps even try reading it in using Pandas to see what happens).

In [ ]:
#this gives us regular expression formatting, allowing us to identify words or patterns
# of letters in input.  You will see this in action below
import re
import pandas as pd
from shapely.geometry import Polygon

#define our function
# this time we are going to pass an argument, which is the file name
def poly_rift(file):
    """Read the fissure swarms in file, and return them as a list of shapely Polygons.

    Each fissure swarm in the file starts with a line beginning with '>'.
    """

    #the 'with' statement here is useful for us, because it automatically closes the file
    # when we exit the statement.  It is saying 'with the file open, do some things, then close'
    # Like a for loop, everything within the 'with' statement needs to be properly indented.
    with open(file) as f:

        #we are going to store all our separate fissures initially as pandas dataframes
        # so let's create the list we are going to put them into here.
        dfs = []

        #once we have the file open, we then need to loop over lines in the file
        # we will use i to keep track of where we are, mainly to know whether this is the 
        # very first line of the file that we have read in
        for i, l in enumerate(f):

            #if you have looked in the file, you would have seen that the problem we have is
            # that the separate fissure swarms are separated by '>' indicators, so we need to 
            # split the file on that 'flag'. To do that, we need to perform a test of whether the line
            # we have read in starts with '>', we can do this using the 'regular expressions' library in Python
            # 're'.  So, we ask 'does the line, l, that we have just read in start with >?', which, communicated
            # as a regular expression reads '^>', where the '^' symbol indicates the start of the line
            if re.match('^>',l):
                #now, if there is a match AND this is NOT the first line we have read in,
                # then we want to store all the previous lines, as they represent a single fissure
                if i != 0:
                    #we use 'pd.DataFrame' to convert our list of lines into a data frame with two columns
                    dfs.append(pd.DataFrame(d, columns=('longitude','latitude')))
                    #convert all values of dataframe to floats i.e. decimal numbers rather than 'strings'
                    dfs[-1] = dfs[-1].astype('float64')
                #reset our list to store the lines representing lon lat coordinates of a given fissure
                d = []
            else:
                #this is the bit of code that gets executed when we read in a 'normal' data row
                #now we have the hard work of converting the line we have read in into something we can store
                # in addition to having lines with '>' characters, this input file has multiple spaces before
                # and between the data.  We want to drop all of this, and just be left with the numbers.  
                # Let's step through what we need to do to clean up this input
                #1. remove the trailing newline character (each line has a character saying 'line end', we don't want to store this)
                l1 = l.rstrip('\n')
                #2. remove all the spaces at the beginning of the line, again we use ^ to indicate the start of the line
                #   the '+' is saying 'find one or more instances of X' (where X in this case is just a space)
                #   the `re.sub(a, b, s)` command replaces 'a' with 'b' in the text string 's'.
                l2 = re.sub('^ +', '', l1)
                #3. now, let's remove all the repeated spaces in the line and leave just one space in each place
                l3 = re.sub(' +', ' ', l2)
                #4. we now have simple lines of numbers, so we just want to split these on the, now, single space
                #   separating them, we just use the '.split()' method that strings have, to split on a particular
                #   character
                l4 = l3.split(' ')
                #(in practice, the string method '.split()' with nothing in the brackets does all four steps at once:
                # it splits on any number of spaces, and ignores spaces and the newline at the start and end.
                # Try replacing the four steps with 'l4 = l.split()' and check you get the same result!)
                
                #finally, we append the line to our list of lines for the current fissure
                # ...dropping the final entry as that is just 0's, so we use our list slicing syntax again
                # where we recall that '-1' indicates the end, which isn't included in the slice by default
                d.append(l4[0:-1])

        #do not forget this step!  The final lines read in need storing,
        # they won't be stored via previous dfs.append statement because it only
        # triggers when a new header is read in and the last lot of data isn't followed by another header (>)
        # it is followed by the end of the file
        dfs.append(pd.DataFrame(d, columns=('longitude','latitude')))
        dfs[-1] = dfs[-1].astype('float64')

    #now, we have all the data read in, we can convert these to polygons so we can plot them
    # this is the same way we did for the glaciers
    polys = []
    for d in dfs:
        polys.append(Polygon(zip(d.longitude, d.latitude)))

    return polys


Now that we have our function to read in the `fisswarms_fil.xy` file, let's run it and plot the fissures, copying our other plotting code from previously.

**Your turn:** the cell below is missing the lines that set up the figure, the axes and the map extent.  Fill them in (look back at the ice cap map), then run it.

In [ ]:
#----------- same as before -----------------------
poly_geoms = poly_jokull()

#Your turn: set up the figure, the axes with a Mercator projection centred on -15,
# and the map extent, just as in the ice cap map above
fig = ___
ax = ___


feature = ctp.feature.NaturalEarthFeature(
    name='coastline', category='physical',
    scale='10m',
    edgecolor='#000000', facecolor='lightgrey')
ax.add_feature(feature, zorder=0)

gl = ax.gridlines(draw_labels=True, linestyle=':', color='k', zorder=-1);
gl.top_labels = False
gl.right_labels = False
#----------- end of same as before -----------------------

#let's update the ice caps to have a zorder of 2, so they plot on top of the rifts
for p in poly_geoms:
    x = ax.add_geometries([p], crs=ccrs.PlateCarree(), facecolor='white', zorder=2)

# run our new function
rift_polys = poly_rift('data/fisswarms_fil.xy')

#as with the ice caps, we loop through each polygon, setting zorder=1 so they plot below the ice caps
# setting the color to that of fresh lava, 'rosybrown'
for p in rift_polys:
    x = ax.add_geometries([p], crs=ccrs.PlateCarree(), facecolor='rosybrown', zorder=1)

## 4.3 Plotting spatial data
***

Now we have our Iceland plot let's add the Holocene eruptions onto it.

In [ ]:
#setup the plot again...
# this time let's store all this in a function so we can skip writing this code each time.
def simp_plot():
    """Draw a map of Iceland with its ice caps and rift zones.

    Returns the figure, axes and gridline objects, so we can change the map later.
    """
    fig = plt.figure(figsize=(12, 6))
    ax = fig.add_subplot(1, 1, 1,
                             projection=ccrs.Mercator(central_longitude=-15))
    ax.set_extent([-25, -13, 63, 67.], crs=ccrs.Geodetic())

    feature = ctp.feature.NaturalEarthFeature(
        name='coastline', category='physical',
        scale='10m',
        edgecolor='#000000', facecolor='lightgrey')
    ax.add_feature(feature, zorder=0)

    gl = ax.gridlines(draw_labels=True, linestyle=':', color='k', zorder=-1);
    gl.top_labels = False
    gl.right_labels = False

    poly_geoms = poly_jokull()

    for p in poly_geoms:
        x = ax.add_geometries([p], crs=ccrs.PlateCarree(), facecolor='white', zorder=2)

    rift_polys = poly_rift('data/fisswarms_fil.xy')
    for p in rift_polys:
        x = ax.add_geometries([p], crs=ccrs.PlateCarree(), facecolor='rosybrown', zorder=1)

    #we want to return the figure, axis, and gridline objects back from our function, 
    # in case we want to update their appearance later.
    return fig, ax, gl
        
#now we call our function to plot, and store the figure and axis objects that it returns
fig, ax, gl = simp_plot()

#now, let's add our data points and use some of the filtering we applied in Exercise 3
df = pd.read_excel('data/iceland_eruptions.xlsx')
#this next command gets us entries with: eruption names, longs, and lats. 
# Then we reduce to unique eruptions and take the mean of all the numerical properties of the dataframe.
# As in Exercise 3, we group by name and age, because some names are used for more than one eruption.
# dropna=False keeps the eruptions whose age is unknown (NaN), which groupby would otherwise drop.
df_clean = df[(df.Eruption.notna()) & (df.Longitude.notna()) & (df.Latitude.notna())].groupby(['Eruption', 'age'], dropna=False, as_index=False).mean()

#let's plot this information
# notice, that when using 'ax.plot()' we need to specify the coordinate system using 'transform', rather than 'crs'
ax.plot(df_clean.Longitude, df_clean.Latitude, 'o', transform=ccrs.Geodetic());

Ok, that tells us where the eruptions are, but it would be nice to get some more information on the plot.  Let's do that by making the symbol size of the eruption scale with the eruption's volume.

We will also add a legend as a key to how the symbol size relates to eruption volume.  We will need to do this by manually setting the size of the symbol in the legend.  It is a bit of a messy solution, but does the job.

In [ ]:
fig, ax, gl = simp_plot()

df = pd.read_excel('data/iceland_eruptions.xlsx')
df_clean = df[(df.Eruption.notna()) & (df.Longitude.notna()) & (df.Latitude.notna())].groupby(['Eruption', 'age'], dropna=False, as_index=False).mean()

#for convenience, this time let's create a new object that just contains the eruptions with volume information
dfvs = df_clean[df_clean['Volume(km3)'].notna()]
#let's be a bit obsessive: it would be good to get the small eruptions to plot last, so
# they aren't obscured by the large eruptions.  There are two options for this, we can use the 'zorder'
# parameter to explicitly control the plotting order of objects when we use matplotlib to plot things,
# or it might be easier to just order the dataframe itself.  We are going to do the latter as Pandas makes this
# easy
dfvs = dfvs.sort_values('Volume(km3)', ascending=False)

#let's plot this information, but scaling symbol size by eruption size.
# ax.scatter() lets us pass a whole column of sizes using 's=', and matches each size up with each point.
# 's' sets the *area* of each symbol (in points squared), so if we make it proportional to the volume,
# the area of each symbol represents the volume of the eruption. (If we made the symbol's diameter
# proportional to the volume instead, its area would grow as volume squared, exaggerating the big eruptions.)
# a scale factor of 25 makes the symbols a sensible size
size_fac = 25
# zorder=3 puts the symbols on top of the land, rifts and ice caps
ax.scatter(dfvs.Longitude, dfvs.Latitude, s=size_fac*dfvs['Volume(km3)'],
           facecolor='orangered', edgecolor='white', zorder=3, transform=ccrs.PlateCarree());

#let's also update the gridline locations, because the auto spacing is a bit useless
# ...an interval of 1.5 degrees in longitude!?  Bizarre.
gl.xlocator = mtckr.MultipleLocator(1)
gl.ylocator = mtckr.MultipleLocator(0.5)

#legend
#for this we need to manually access the plotting elements that allows
# us to create circles, which we do by importing 'Line2D'
from matplotlib.lines import Line2D

#now, we create a list of symbols using Line2D, one for each volume we want in the legend.
# Look up the manual page for Line2D to see all its functionality.
# For now we are concerned in setting the marker as 'o', i.e., a circle,
# and setting the line style, ls, to '', which means we don't want a line drawn.
# Line2D's 'markersize' is the symbol's diameter rather than its area, so we take the square root
# of the area we gave scatter() above.
legend_vols = [1, 10, 20, 30]
legend_elements = [Line2D([0], [0], marker='o', ls='', label=f'{v} km$^3$', markeredgecolor='white',
                          markerfacecolor='orangered', markersize=(size_fac*v)**0.5)
                   for v in legend_vols]

ax.legend(handles=legend_elements, loc='right')

#final finishing touch
ax.set_title('Iceland Holocene eruptions');

**Your turn:** how many of the eruptions in `dfvs` have a volume of more than 10 km³?

In [ ]:
n_big = ___
check('big eruptions', n_big)

# Independent coding
***

> **Aim:** To read in and plot earthquake data from 2019 on the continental USA.

Often one of the first tasks when you have a geophysical or geochemical dataset is to get a sense of where your observations are located spatially.  Are there gaps in the data coverage?  Biases in locations that have been sampled?  All this can be quickly evaluated with a map.  Here we are going to look at earthquake locations from the continental US, __[downloaded from the USGS](https://earthquake.usgs.gov/earthquakes/search/)__, and stored in the data folder as `us_eqs.csv`.  The file contains all 3,472 earthquakes of magnitude 2.5 or more between 1 January and 9 September 2019.

Some of the 'to-do's' in this exercise go a little beyond the code examples presented above.  This is a good opportunity to hone your search engine skills to find the answer online, pointers are also given to a couple of online examples that will help.

Work through these checkpoints:

1. **Set up.** Create a new notebook, called `Exercise4_solution`, in the course folder.  Import the libraries you need.
1. **Read in.** Read the datafile `us_eqs.csv` into a dataframe called `df`, and find the columns you need for plotting.
   Check: `len(df)` should be 3472, and `df.mag.max()` should be about 7.1.
1. **The map.** Make a map of the continental USA using the Albers Equal Area projection (_Hint: __[Cartopy projections](https://cartopy.readthedocs.io/stable/reference/projections.html)__)_.
1. **State boundaries.** Plot on the state boundaries (_Hint: always make good use of __[online examples](https://cartopy.readthedocs.io/stable/gallery/lines_and_polygons/hurricane_katrina.html)__ when writing your code_).
1. **Earthquakes.** Plot on the earthquake locations.  You should see the most earthquakes in California and Nevada, with clusters in Oklahoma and around Yellowstone.
1. **Save** your map to a file, e.g. `fig.savefig('exercise4_earthquakes.png', dpi=300)`.

Once you have the earthquakes plotted, consider why they are distributed where they are.  A good place to start thinking about this is on the __[USGS website](https://www.usgs.gov/natural-hazards/earthquake-hazards/hazards)__.

### Plot optimisation

This gives us the basic plot, but we can try a few other things (see the hints below for some pointers if you are stuck).

- Can you colour the points by depth, and scale their size by magnitude?
- ...plotting the largest magnitude earthquakes first so they don't obscure the smaller events,
- ...and provide a colour bar for the depth colour scale?

_Hint 1: As we did for the eruption volumes above, you can give `ax.scatter()` whole columns of the dataframe, so you don't need a loop. Use `s=` for the sizes, and remember that it sets the area of each symbol. Magnitudes are small numbers, so you will need to transform them to get a sensible range of symbol sizes._

_Hint 2: `ax.scatter()` can also colour each point for you: pass the depths with `c=`, and choose a colour map with `cmap=` (pick one of __[these](https://matplotlib.org/stable/users/explain/colors/colormaps.html)__, such as `'viridis'`)._

_Hint 3: `ax.scatter()` returns an object that you can store, for example `im = ax.scatter(...)`, and then pass to `fig.colorbar(im, ax=ax, label=...)` to add a colour scale, just like we did in Exercise 3._